In [1]:
import pandas as pd

df = pd.read_csv('../data/phishing_url_dataset_raw.csv')
df.shape

(235795, 55)

In [3]:
# We drop the five redundant columns from S1-02
df_numeric = df.select_dtypes(include='number')
df_numeric = df_numeric.drop(['DomainTitleMatchScore', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'HasObfuscation'], axis=1)

In [4]:
# Split the dataset into inputs (X) and (y), then divide both into 80% training set and 20% test set, while preserving same ratio (phishing vs legitimate)
X = df_numeric.drop('label', axis=1)
y = df_numeric['label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
# Train the final model on the full training set (same as S1-03)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [6]:
from sklearn.metrics import classification_report

# Generate precision, recall, and F1 score per class - more informative than raw accuracy  
print(classification_report(y_test, y_pred, target_names=['Phishing', 'Legitimate']))

              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00     20189
  Legitimate       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



In [12]:
import joblib
import numpy as np


# Save the model to disk, reload it, and confirm it predicts identically to the original
joblib.dump(rf, '../backend/app/ml/phishguard_model.pkl')

loaded_model = joblib.load('../backend/app/ml/phishguard_model.pkl')

original_preds = rf.predict(X_test)
loaded_preds = loaded_model.predict(X_test)
print(f"Predictions match: {np.array_equal(original_preds, loaded_preds)}")

Predictions match: True


In [13]:
# # Save the model's feature column names to disk, in the right order, for reuse later
feature_columns = X_train.columns.tolist()
joblib.dump(feature_columns, '../backend/app/ml/feature_columns.pkl')

['../backend/app/ml/feature_columns.pkl']